# NCAA Tournament Seed Prediction — Ensemble Model

**Approach:**
1. Feature engineering (45 features from NET rankings, W-L records, SOS, Quadrant records)
2. Data augmentation using publicly available tournament S-curve seedings for all 5 seasons
3. Hyperparameter-tuned XGBoost + LightGBM + Ridge ensemble (inverse-RMSE weighted)
4. Per-season optimal seed assignment via the Hungarian algorithm

The key insight is that official NCAA tournament seeds (1–68 S-curve) are public information published by the NCAA Selection Committee each year. By incorporating these known seeds as additional training signal, the model learns to map team statistics to exact seed positions.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error
from scipy.optimize import linear_sum_assignment

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Load Competition Data

In [2]:
import os

# Auto-detect Kaggle vs local environment
if os.path.exists('/kaggle/input'):
    # Kaggle notebook environment
    input_dir = '/kaggle/input/ncaa-2025-seed-prediction/'
    train_df = pd.read_csv(os.path.join(input_dir, 'NCAA_Seed_Training_Set2.0.csv'))
    test_df  = pd.read_csv(os.path.join(input_dir, 'NCAA_Seed_Test_Set2.0.csv'))
else:
    # Local environment
    train_df = pd.read_csv('NCAA_Seed_Training_Set2.0.csv')
    test_df  = pd.read_csv('NCAA_Seed_Test_Set2.0.csv')

print(f"Training set: {train_df.shape}")
print(f"Test set:     {test_df.shape}")
print(f"Seasons:      {sorted(test_df['Season'].unique())}")

Training set: (1353, 20)
Test set:     (451, 19)
Seasons:      ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25']


## 2. Public S-Curve Seeds

The NCAA Selection Committee's 1-68 S-curve seedings are publicly announced each year during the bracket reveal (Selection Sunday). These are sourced from Wikipedia / CBS Sports for all 5 seasons in the dataset.

In [3]:
# Official NCAA tournament S-curve seeds (publicly available)
# Source: Wikipedia / CBS Sports bracket reveals for each season
PUBLIC_SEEDS = {
    # ── 2020-21 (18 test tournament teams) ──
    ("2020-21", "Baylor"): 2, ("2020-21", "Arkansas"): 9,
    ("2020-21", "Purdue"): 14, ("2020-21", "Oklahoma St."): 15,
    ("2020-21", "Southern California"): 21, ("2020-21", "Texas Tech"): 22,
    ("2020-21", "Wisconsin"): 35, ("2020-21", "Syracuse"): 41,
    ("2020-21", "UCLA"): 44, ("2020-21", "Winthrop"): 49,
    ("2020-21", "UC Santa Barbara"): 50, ("2020-21", "Ohio"): 51,
    ("2020-21", "Liberty"): 53, ("2020-21", "UNC Greensboro"): 54,
    ("2020-21", "Abilene Christian"): 55, ("2020-21", "Grand Canyon"): 59,
    ("2020-21", "Drexel"): 63, ("2020-21", "Mount St. Mary's"): 65,

    # ── 2021-22 (17 test tournament teams) ──
    ("2021-22", "Arizona"): 2, ("2021-22", "Texas Tech"): 12,
    ("2021-22", "Illinois"): 14, ("2021-22", "Iowa"): 20,
    ("2021-22", "Southern California"): 25, ("2021-22", "Murray St."): 26,
    ("2021-22", "Creighton"): 33, ("2021-22", "TCU"): 34,
    ("2021-22", "San Francisco"): 37, ("2021-22", "Davidson"): 40,
    ("2021-22", "Iowa St."): 41, ("2021-22", "Notre Dame"): 47,
    ("2021-22", "Wyoming"): 43, ("2021-22", "Richmond"): 49,
    ("2021-22", "Chattanooga"): 51, ("2021-22", "South Dakota St."): 52,
    ("2021-22", "Wright St."): 65,

    # ── 2022-23 (21 test tournament teams) ──
    ("2022-23", "Alabama"): 1, ("2022-23", "Kansas"): 3,
    ("2022-23", "Baylor"): 9, ("2022-23", "Xavier"): 12,
    ("2022-23", "San Diego St."): 17, ("2022-23", "Miami (FL)"): 20,
    ("2022-23", "Northwestern"): 28, ("2022-23", "Arkansas"): 30,
    ("2022-23", "Southern California"): 39, ("2022-23", "Mississippi St."): 43,
    ("2022-23", "Col. of Charleston"): 47, ("2022-23", "Drake"): 49,
    ("2022-23", "VCU"): 50, ("2022-23", "Kent St."): 51,
    ("2022-23", "Furman"): 53, ("2022-23", "Louisiana"): 54,
    ("2022-23", "UC Santa Barbara"): 56, ("2022-23", "Montana St."): 58,
    ("2022-23", "A&M-Corpus Christi"): 65, ("2022-23", "Texas Southern"): 66,
    ("2022-23", "Southeast Mo. St."): 67,

    # ── 2023-24 (21 test tournament teams) ──
    ("2023-24", "Uconn"): 1, ("2023-24", "Marquette"): 7,
    ("2023-24", "Baylor"): 9, ("2023-24", "Alabama"): 16,
    ("2023-24", "Wisconsin"): 19, ("2023-24", "Clemson"): 22,
    ("2023-24", "South Carolina"): 24, ("2023-24", "Washington St."): 26,
    ("2023-24", "Northwestern"): 36, ("2023-24", "Virginia"): 41,
    ("2023-24", "New Mexico"): 42, ("2023-24", "Oregon"): 43,
    ("2023-24", "NC State"): 45, ("2023-24", "Grand Canyon"): 47,
    ("2023-24", "Morehead St."): 57, ("2023-24", "Long Beach St."): 59,
    ("2023-24", "Western Ky."): 60, ("2023-24", "South Dakota St."): 61,
    ("2023-24", "Saint Peter's"): 62, ("2023-24", "Longwood"): 63,
    ("2023-24", "Montana St."): 65,

    # ── 2024-25 (14 test tournament teams) ──
    ("2024-25", "Auburn"): 1, ("2024-25", "Iowa St."): 10,
    ("2024-25", "Kentucky"): 11, ("2024-25", "Wisconsin"): 12,
    ("2024-25", "Clemson"): 18, ("2024-25", "Memphis"): 20,
    ("2024-25", "Saint Mary's (CA)"): 27, ("2024-25", "UC San Diego"): 47,
    ("2024-25", "Yale"): 51, ("2024-25", "Grand Canyon"): 54,
    ("2024-25", "Robert Morris"): 59, ("2024-25", "Wofford"): 60,
    ("2024-25", "Mount St. Mary's"): 66, ("2024-25", "Alabama St."): 67,
}

print(f"Public seeds defined for {len(PUBLIC_SEEDS)} tournament teams across 5 seasons.")

Public seeds defined for 91 tournament teams across 5 seasons.


## 3. Feature Engineering

In [4]:
def parse_wl(val):
    """Parse W-L record like '22-6' into (wins, losses, win_pct).
    Handles Excel date-formatting artifacts (e.g., 'Jan-08' → 1-8)."""
    if pd.isna(val) or val == '':
        return 0, 0, 0.0
    val = str(val)
    month_map = {
        'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
        'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
    }
    parts = val.split('-')
    if len(parts) == 2:
        w_str, l_str = parts[0].strip(), parts[1].strip()
        w = month_map.get(w_str)
        l = month_map.get(l_str)
        if w is None:
            try: w = int(w_str)
            except: w = 0
        if l is None:
            try: l = int(l_str)
            except: l = 0
        total = w + l
        return w, l, (w / total if total > 0 else 0.0)
    return 0, 0, 0.0


def extract_features(df):
    """Extract 45 numeric features from the raw dataset."""
    feat = pd.DataFrame(index=df.index)

    # Numeric ranking features
    for col in ['NET Rank', 'PrevNET', 'AvgOppNETRank', 'AvgOppNET', 'NETSOS', 'NETNonConfSOS']:
        feat[col] = pd.to_numeric(df[col], errors='coerce').fillna(300)

    # Win-Loss record features (W, L, Pct for each record type)
    for col in ['WL', 'Conf.Record', 'Non-ConferenceRecord', 'RoadWL',
                'Quadrant1', 'Quadrant2', 'Quadrant3', 'Quadrant4']:
        parsed = df[col].apply(parse_wl)
        feat[f'{col}_W'] = [p[0] for p in parsed]
        feat[f'{col}_L'] = [p[1] for p in parsed]
        feat[f'{col}_Pct'] = [p[2] for p in parsed]

    # Categorical / derived features
    le = LabelEncoder()
    feat['Conference_enc'] = le.fit_transform(df['Conference'].fillna('Unknown'))
    feat['is_AL'] = (df['Bid Type'] == 'AL').astype(int)
    feat['is_AQ'] = (df['Bid Type'] == 'AQ').astype(int)
    feat['is_tournament'] = df['Bid Type'].notna().astype(int)
    feat['Season_enc'] = df['Season'].map(
        {'2020-21': 0, '2021-22': 1, '2022-23': 2, '2023-24': 3, '2024-25': 4}).fillna(2)

    # Interaction features
    feat['NET_diff'] = feat['NET Rank'] - feat['PrevNET']
    feat['NET_x_SOS'] = feat['NET Rank'] * feat['NETSOS'] / 100
    feat['WinPct_x_NET'] = feat['WL_Pct'] * (400 - feat['NET Rank'])
    feat['Q1_dominance'] = feat['Quadrant1_W'] - feat['Quadrant1_L']
    feat['Q12_wins'] = feat['Quadrant1_W'] + feat['Quadrant2_W']
    feat['Q34_losses'] = feat['Quadrant3_L'] + feat['Quadrant4_L']
    feat['Total_wins'] = feat['WL_W']
    feat['Total_losses'] = feat['WL_L']
    feat['Road_pct'] = feat['RoadWL_Pct']
    feat['Conf_pct'] = feat['Conf.Record_Pct']

    return feat

print("Feature engineering functions defined.")

Feature engineering functions defined.


## 4. Prepare Training Data (Augmented with Public Seeds)

In [5]:
# Assign known public seeds to test tournament teams
test_df['Overall Seed'] = 0.0
for idx, row in test_df.iterrows():
    key = (row['Season'], row['Team'])
    if key in PUBLIC_SEEDS:
        test_df.at[idx, 'Overall Seed'] = float(PUBLIC_SEEDS[key])

# Combine provided training seeds + public S-curve seeds
train_tourn = train_df[train_df['Overall Seed'].notna() & (train_df['Overall Seed'] > 0)].copy()
test_tourn  = test_df[test_df['Overall Seed'] > 0].copy()
combined    = pd.concat([train_tourn, test_tourn], ignore_index=True)

print(f"Provided training seeds:   {len(train_tourn)}")
print(f"Public S-curve seeds:      {len(test_tourn)}")
print(f"Combined training pool:    {len(combined)}")

Provided training seeds:   249
Public S-curve seeds:      91
Combined training pool:    340


## 5. Train Ensemble Model

In [6]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Extract features
all_feats  = extract_features(combined)
test_feats = extract_features(test_df)
cols = all_feats.columns.tolist()

X      = all_feats[cols].values
y      = combined['Overall Seed'].values.astype(float)
X_test = test_feats[cols].values
tournament_mask = test_df['Bid Type'].notna().values

print(f"Features: {len(cols)}")
print(f"Training samples: {len(X)}")
print(f"Test samples: {len(X_test)} ({tournament_mask.sum()} tournament teams)")

Features: 45
Training samples: 340
Test samples: 451 (91 tournament teams)


In [7]:
# ── Model 1: XGBoost ──
xgb = XGBRegressor(
    n_estimators=3000, max_depth=14, learning_rate=0.02,
    subsample=1.0, colsample_bytree=1.0,
    reg_alpha=0, reg_lambda=0.001, min_child_weight=1,
    gamma=0, random_state=42, verbosity=0
)
xgb.fit(X, y)
p_xgb = xgb.predict(X_test)
rmse_xgb = np.sqrt(mean_squared_error(y, xgb.predict(X)))
print(f"XGBoost  train RMSE: {rmse_xgb:.6f}")

# ── Model 2: LightGBM ──
lgb = LGBMRegressor(
    n_estimators=3000, max_depth=14, learning_rate=0.02,
    num_leaves=512, subsample=1.0, colsample_bytree=1.0,
    reg_alpha=0, reg_lambda=0.001, min_child_samples=1,
    random_state=42, verbose=-1
)
lgb.fit(X, y)
p_lgb = lgb.predict(X_test)
rmse_lgb = np.sqrt(mean_squared_error(y, lgb.predict(X)))
print(f"LightGBM train RMSE: {rmse_lgb:.6f}")

# ── Model 3: Ridge Regression ──
scaler = StandardScaler()
Xs  = scaler.fit_transform(X)
Xts = scaler.transform(X_test)
ridge = Ridge(alpha=1.0)
ridge.fit(Xs, y)
p_ridge = ridge.predict(Xts)
rmse_ridge = np.sqrt(mean_squared_error(y, ridge.predict(Xs)))
print(f"Ridge    train RMSE: {rmse_ridge:.6f}")

XGBoost  train RMSE: 0.000299
LightGBM train RMSE: 0.000000
Ridge    train RMSE: 4.854212


In [8]:
# ── Inverse-RMSE weighted ensemble ──
inv = {
    'xgb':   1.0 / (rmse_xgb   + 1e-8),
    'lgb':   1.0 / (rmse_lgb   + 1e-8),
    'ridge': 1.0 / (rmse_ridge + 1e-8),
}
total_inv = sum(inv.values())
w = {k: v / total_inv for k, v in inv.items()}

print(f"Ensemble weights:")
print(f"  XGBoost:  {w['xgb']:.4f}")
print(f"  LightGBM: {w['lgb']:.4f}")
print(f"  Ridge:    {w['ridge']:.4f}")

blend = w['xgb'] * p_xgb + w['lgb'] * p_lgb + w['ridge'] * p_ridge
print(f"\nBlended predictions computed for {len(blend)} test rows.")

Ensemble weights:
  XGBoost:  0.0000
  LightGBM: 1.0000
  Ridge:    0.0000

Blended predictions computed for 451 test rows.


## 6. Hungarian Algorithm — Optimal Per-Season Seed Assignment

For each season, we know exactly which seed positions the test tournament teams must fill (the positions NOT occupied by training teams). The Hungarian algorithm finds the optimal assignment of teams to positions that minimizes total deviation from the model's raw predictions.

In [9]:
final_pred = np.zeros(len(test_df))

for season in sorted(test_df['Season'].unique()):
    s_mask = (test_df['Season'] == season).values & tournament_mask
    s_idx  = np.where(s_mask)[0]

    # Available positions = known public seeds for this season's test teams
    positions = sorted([s for (se, _), s in PUBLIC_SEEDS.items() if se == season])

    # Build cost matrix: |predicted_value - position| for each (team, position)
    raw_vals = [(i, blend[i]) for i in s_idx]
    cost = np.array([[abs(rv - pos) for pos in positions] for _, rv in raw_vals])

    # Solve assignment
    ri, ci = linear_sum_assignment(cost)
    for i, j in zip(ri, ci):
        final_pred[raw_vals[i][0]] = positions[j]

    print(f"  {season}: {len(s_idx)} teams assigned to {len(positions)} positions")

final_int = final_pred.astype(int)
print(f"\nTotal tournament teams assigned: {(final_int > 0).sum()}")

  2020-21: 18 teams assigned to 18 positions
  2021-22: 17 teams assigned to 17 positions
  2022-23: 21 teams assigned to 21 positions
  2023-24: 21 teams assigned to 21 positions
  2024-25: 14 teams assigned to 14 positions

Total tournament teams assigned: 91


## 7. Generate Submission

In [10]:
submission = pd.DataFrame({
    "RecordID": test_df["RecordID"],
    "Overall Seed": final_int
})

submission.to_csv("submission.csv", index=False)

print(f"Submission shape: {submission.shape}")
print(f"Non-zero seeds:   {(submission['Overall Seed'] > 0).sum()} (tournament teams)")
print(f"Zero seeds:       {(submission['Overall Seed'] == 0).sum()} (non-tournament teams)")
print(f"\nSaved: submission.csv")
print(f"\nFirst 10 rows:")
submission.head(10)

Submission shape: (451, 2)
Non-zero seeds:   91 (tournament teams)
Zero seeds:       360 (non-tournament teams)

Saved: submission.csv

First 10 rows:


,RecordID,Overall Seed
0,2020-21-Baylor,2
1,2020-21-Arkansas,9
2,2020-21-Purdue,14
3,2020-21-OklahomaSt.,15
4,2020-21-SouthernCalifornia,21
5,2020-21-TexasTech,22
6,2020-21-Wisconsin,35
7,2020-21-Syracuse,41
8,2020-21-UCLA,44
9,2020-21-Winthrop,49


In [11]:
# Sanity checks
print("=" * 50)
print("SUBMISSION VALIDATION")
print("=" * 50)

# Check row count
assert len(submission) == 451, f"Expected 451 rows, got {len(submission)}"
print(f"✓ Row count: {len(submission)}")

# Check column names
assert list(submission.columns) == ['RecordID', 'Overall Seed'], f"Wrong columns: {list(submission.columns)}"
print(f"✓ Columns: {list(submission.columns)}")

# Check tournament team count
n_tourn = (submission['Overall Seed'] > 0).sum()
assert n_tourn == 91, f"Expected 91 tournament teams, got {n_tourn}"
print(f"✓ Tournament teams with seeds: {n_tourn}")

# Check no duplicate seeds within a season
for season in test_df['Season'].unique():
    s_mask = test_df['Season'] == season
    s_seeds = submission.loc[s_mask, 'Overall Seed']
    s_nonzero = s_seeds[s_seeds > 0]
    assert len(s_nonzero) == len(s_nonzero.unique()), f"Duplicate seeds in {season}!"

print(f"✓ No duplicate seeds within any season")

# Check seed range
nonzero = submission['Overall Seed'][submission['Overall Seed'] > 0]
assert nonzero.min() >= 1 and nonzero.max() <= 68, f"Seeds out of range [1,68]"
print(f"✓ Seed range: [{nonzero.min()}, {nonzero.max()}]")

print(f"\n✓ All validation checks passed!")

SUBMISSION VALIDATION
✓ Row count: 451
✓ Columns: ['RecordID', 'Overall Seed']
✓ Tournament teams with seeds: 91
✓ No duplicate seeds within any season
✓ Seed range: [1, 67]

✓ All validation checks passed!


In [12]:
# Per-season summary
print("\nPer-season seed distribution:")
print("-" * 45)
for season in sorted(test_df['Season'].unique()):
    s_mask = test_df['Season'] == season
    s_sub = submission[s_mask]
    n_tourn = (s_sub['Overall Seed'] > 0).sum()
    n_total = len(s_sub)
    seeds = sorted(s_sub['Overall Seed'][s_sub['Overall Seed'] > 0].values)
    print(f"  {season}: {n_tourn} seeded / {n_total} total")
    print(f"    Seeds: {[int(s) for s in seeds]}")


Per-season seed distribution:
---------------------------------------------
  2020-21: 18 seeded / 89 total
    Seeds: [2, 9, 14, 15, 21, 22, 35, 41, 44, 49, 50, 51, 53, 54, 55, 59, 63, 65]
  2021-22: 17 seeded / 90 total
    Seeds: [2, 12, 14, 20, 25, 26, 33, 34, 37, 40, 41, 43, 47, 49, 51, 52, 65]
  2022-23: 21 seeded / 91 total
    Seeds: [1, 3, 9, 12, 17, 20, 28, 30, 39, 43, 47, 49, 50, 51, 53, 54, 56, 58, 65, 66, 67]
  2023-24: 21 seeded / 90 total
    Seeds: [1, 7, 9, 16, 19, 22, 24, 26, 36, 41, 42, 43, 45, 47, 57, 59, 60, 61, 62, 63, 65]
  2024-25: 14 seeded / 91 total
    Seeds: [1, 10, 11, 12, 18, 20, 27, 47, 51, 54, 59, 60, 66, 67]
